# Cálculo Manual de un Forward y Backward Pass

## Configuración Inicial

En esta sección realizaremos los siguientes pasos:
- Cargar las librerías necesarias (NumPy).
- Cargar una imagen de MNIST desde los datos de entrenamiento.
- Cargar los pesos no entrenados de la red neuronal desde el archivo `mlp_weights.npz`.

Esto nos permitirá tener todos los datos preparados para realizar el cálculo manual del forward pass (propagación hacia adelante) y backward pass (retropropagación).

In [1]:
import numpy as np
import sys
import os

# Añadir la ruta raíz del proyecto al path de Python
project_root = os.path.abspath(os.path.join(os.getcwd(), '../'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Importar funciones de MNIST
from proyecto2_ann.data.mnist_loader import load_mnist

# Importar funciones de activación
from proyecto2_ann.neural_network.activations import relu, relu_derivative, softmax

# Importar funciones de pérdida
from proyecto2_ann.neural_network.loss import one_hot, cross_entropy_loss

print('✓ Librerías importadas correctamente')

# Cargar datos de MNIST
print('\nCargando datos de MNIST...')
X_train, y_train, X_test, y_test = load_mnist()
print(f'✓ Datos de MNIST cargados')
print(f'  - X_train shape: {X_train.shape}')
print(f'  - y_train shape: {y_train.shape}')

# Seleccionar la primera imagen de entrenamiento
x_sample = X_train[0]
y_sample = y_train[0]

print(f'\n--- Imagen Seleccionada ---')
print(f'Etiqueta de la primera imagen: {y_sample}')
print(f'Shape de la imagen: {x_sample.shape}')
print(f'Valor mínimo: {x_sample.min():.6f}')
print(f'Valor máximo: {x_sample.max():.6f}')

# Cargar pesos no entrenados
print(f'\nCargando pesos no entrenados...')
pesos_path = os.path.join(project_root, 'proyecto2_ann/models/mlp_weights.npz')
pesos = np.load(pesos_path)

# Extraer matrices de pesos y sesgos
W1 = pesos['W1']
b1 = pesos['b1']
W2 = pesos['W2']
b2 = pesos['b2']

print(f'✓ Pesos cargados desde {pesos_path}')

# Imprimir formas para verificación
print(f'\n--- Formas de los Pesos y Sesgos ---')
print(f'W1 shape: {W1.shape}  (capas ocultas × entrada)')
print(f'b1 shape: {b1.shape}  (sesgos capa oculta)')
print(f'W2 shape: {W2.shape}  (salida × capas ocultas)')
print(f'b2 shape: {b2.shape}  (sesgos capa salida)')

print(f'\n✓ ¡Configuración completada! Listo para comenzar el cálculo manual.')

✓ Librerías importadas correctamente

Cargando datos de MNIST...
MNIST cargado: train=(60000, 784), test=(10000, 784)
✓ Datos de MNIST cargados
  - X_train shape: (60000, 784)
  - y_train shape: (60000,)

--- Imagen Seleccionada ---
Etiqueta de la primera imagen: 5
Shape de la imagen: (784,)
Valor mínimo: 0.000000
Valor máximo: 1.000000

Cargando pesos no entrenados...
✓ Pesos cargados desde /home/gerardo/ia/neuronas/proyecto2_ann/models/mlp_weights.npz

--- Formas de los Pesos y Sesgos ---
W1 shape: (64, 784)  (capas ocultas × entrada)
b1 shape: (64,)  (sesgos capa oculta)
W2 shape: (10, 64)  (salida × capas ocultas)
b2 shape: (10,)  (sesgos capa salida)

✓ ¡Configuración completada! Listo para comenzar el cálculo manual.


## Fase 1: Propagación Hacia Adelante (Forward Pass)

Ahora que tenemos la imagen de entrada (`x_sample`) y los pesos (`W1`, `b1`, `W2`, `b2`), podemos comenzar el recorrido hacia adelante.

1.  **Capa Oculta**: Calcularemos la salida de la capa oculta, que implica un producto punto seguido de una función de activación ReLU.
2.  **Capa de Salida**: Usaremos la salida de la capa oculta para calcular el resultado final de la red, aplicando un producto punto y una función de activación Softmax.

### 1.1. Capa Oculta (Entrada -> Oculta)

In [2]:
# Calcular la salida ponderada de la capa oculta
# Z1 = W1 @ x_sample + b1
Z1 = np.dot(W1, x_sample) + b1

# Aplicar la función de activación ReLU
# A1 = relu(Z1)
A1 = relu(Z1)

print("--- Capa Oculta ---")
print(f"Shape de Z1 (salida ponderada): {Z1.shape}")
print(f"Primeros 5 valores de Z1: {Z1[:5]}")
print("\n")
print(f"Shape de A1 (salida activada): {A1.shape}")
print(f"Primeros 5 valores de A1: {A1[:5]}")

--- Capa Oculta ---
Shape de Z1 (salida ponderada): (64,)
Primeros 5 valores de Z1: [ 1.26329089  5.71840769 -4.15973235 -6.72640884 -1.33585876]


Shape de A1 (salida activada): (64,)
Primeros 5 valores de A1: [1.26329089 5.71840769 0.         0.         0.        ]


### 1.2. Capa de Salida (Oculta -> Salida)

In [3]:
# Calcular la salida ponderada de la capa de salida
# Z2 = W2 @ A1 + b2
Z2 = np.dot(W2, A1) + b2

# Aplicar la función de activación Softmax para obtener probabilidades
# A2 = softmax(Z2)
A2 = softmax(Z2)

prediccion = np.argmax(A2)

print("--- Capa de Salida ---")
print(f"Shape de Z2 (salida ponderada): {Z2.shape}")
print(f"Valores de Z2: {Z2}")
print("\n")
print(f"Shape de A2 (probabilidades): {A2.shape}")
print(f"Valores de A2 (probabilidades por clase):\n{A2}")
print("\n")
print(f"Suma de probabilidades de A2: {np.sum(A2):.6f} (debe ser ~1.0)")
print(f"\nPredicción de la red (clase con mayor probabilidad): {prediccion}")
print(f"Etiqueta real: {y_sample}")

--- Capa de Salida ---
Shape de Z2 (salida ponderada): (10,)
Valores de Z2: [-13.27465246  -5.68430425   3.87658092  36.51446909 -36.80855769
  39.70995471 -17.77013173  -1.82909051  -7.48837682  -5.60635088]


Shape de A2 (probabilidades): (10,)
Valores de A2 (probabilidades por clase):
[9.36804595e-24 1.85393864e-20 2.63229784e-16 3.93359608e-02
 5.63642635e-34 9.60664039e-01 1.04541126e-25 8.75776748e-19
 3.05208480e-21 2.00424161e-20]


Suma de probabilidades de A2: 1.000000 (debe ser ~1.0)

Predicción de la red (clase con mayor probabilidad): 5
Etiqueta real: 5


## Fase 2: Cálculo del Error (Loss)

Con la predicción (`A2`) y la etiqueta real (`y_sample`), podemos calcular qué tan equivocada estuvo la red. Para ello, usamos la **función de pérdida de entropía cruzada categórica** (Categorical Cross-Entropy Loss).

Primero, convertimos la etiqueta numérica a un formato `one-hot`.

In [4]:
# Convertir la etiqueta a formato one-hot
y_one_hot = one_hot(y_sample, num_classes=10)

# Calcular la pérdida
loss = cross_entropy_loss(A2, y_one_hot)

print("--- Cálculo del Error ---")
print(f"Etiqueta real: {y_sample}")
print(f"Etiqueta en formato one-hot: {y_one_hot}")
print(f"\nValor de la pérdida (Cross-Entropy): {loss:.6f}")

--- Cálculo del Error ---
Etiqueta real: 5
Etiqueta en formato one-hot: [0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]

Valor de la pérdida (Cross-Entropy): 0.040131


## Fase 3: Retropropagación (Backward Pass)

El objetivo de la retropropagación es calcular los **gradientes**, que nos dicen cómo debemos ajustar cada peso y sesgo para reducir el error. Lo hacemos propagando el error hacia atrás, desde la capa de salida hasta la capa de entrada.

1.  **Capa de Salida**: Calculamos los gradientes `dW2` y `db2`.
2.  **Capa Oculta**: Calculamos los gradientes `dW1` y `db1`.

### 3.1. Gradientes de la Capa de Salida

In [5]:
# Gradiente del error con respecto a la salida ponderada Z2
# dZ2 = A2 - y_one_hot
dZ2 = A2 - y_one_hot

# Gradiente para los pesos W2
# dW2 = dZ2 ⊗ A1^T (producto exterior)
dW2 = np.outer(dZ2, A1)

# Gradiente para el sesgo b2
db2 = dZ2

print("--- Gradientes de la Capa de Salida ---")
print(f"Shape de dZ2: {dZ2.shape}")
print(f"Shape de dW2: {dW2.shape} (debe coincidir con W2: {W2.shape})")
print(f"Shape de db2: {db2.shape} (debe coincidir con b2: {b2.shape})")

--- Gradientes de la Capa de Salida ---
Shape de dZ2: (10,)
Shape de dW2: (10, 64) (debe coincidir con W2: (10, 64))
Shape de db2: (10,) (debe coincidir con b2: (10,))


### 3.2. Gradientes de la Capa Oculta

In [6]:
# Propagar el gradiente hacia la capa oculta
# dA1 = W2^T @ dZ2
dA1 = np.dot(W2.T, dZ2)

# Gradiente a través de la activación ReLU
# dZ1 = dA1 * relu'(Z1)
dZ1 = dA1 * relu_derivative(Z1)

# Gradiente para los pesos W1
# dW1 = dZ1 ⊗ x_sample^T (producto exterior)
dW1 = np.outer(dZ1, x_sample)

# Gradiente para el sesgo b1
db1 = dZ1

print("--- Gradientes de la Capa Oculta ---")
print(f"Shape de dA1: {dA1.shape}")
print(f"Shape de dZ1: {dZ1.shape}")
print(f"Shape de dW1: {dW1.shape} (debe coincidir con W1: {W1.shape})")
print(f"Shape de db1: {db1.shape} (debe coincidir con b1: {b1.shape})")

--- Gradientes de la Capa Oculta ---
Shape de dA1: (64,)
Shape de dZ1: (64,)
Shape de dW1: (64, 784) (debe coincidir con W1: (64, 784))
Shape de db1: (64,) (debe coincidir con b1: (64,))


## Fase 4: Actualización de Pesos

Finalmente, usamos los gradientes calculados para actualizar los pesos y sesgos de la red. Este es el paso de "aprendizaje" real, donde la red ajusta sus parámetros para mejorar su rendimiento en la siguiente iteración.

La fórmula es: `nuevo_peso = peso_anterior - tasa_de_aprendizaje * gradiente`.

In [7]:
# Definir una tasa de aprendizaje (learning rate)
lr = 0.01

# Calcular los nuevos pesos y sesgos
W1_new = W1 - lr * dW1
b1_new = b1 - lr * db1
W2_new = W2 - lr * dW2
b2_new = b2 - lr * db2

print("--- Actualización de Pesos ---")
print(f"Tasa de aprendizaje (lr): {lr}")
print("\nLos nuevos pesos (W1_new, b1_new, W2_new, b2_new) han sido calculados.")
print("Estos serían los pesos que se usarían para la siguiente iteración de entrenamiento.")

# Opcional: verificar la diferencia
print(f"\nDiferencia en W1 (norma): {np.linalg.norm(W1 - W1_new)}")
print(f"Diferencia en W2 (norma): {np.linalg.norm(W2 - W2_new)}")

--- Actualización de Pesos ---
Tasa de aprendizaje (lr): 0.01

Los nuevos pesos (W1_new, b1_new, W2_new, b2_new) han sido calculados.
Estos serían los pesos que se usarían para la siguiente iteración de entrenamiento.

Diferencia en W1 (norma): 0.016935559210948976
Diferencia en W2 (norma): 0.009256725977430522


In [8]:
import pandas as pd
import os

# --- EXPORTAR A EXCEL ---

# Definir el nombre del archivo de salida
excel_path = os.path.join(project_root, 'document/calculo_manual.xlsx')

# Usar un ExcelWriter para guardar múltiples DataFrames en diferentes hojas
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    print(f"Creando archivo Excel en: {excel_path}")

    # --- Fase 0: Entradas y Pesos Iniciales ---
    pd.DataFrame(x_sample.reshape(-1, 1), columns=['valor']).to_excel(writer, sheet_name='0_Entrada_X', index_label='Pixel_ID')
    pd.DataFrame(W1).to_excel(writer, sheet_name='0_Peso_W1')
    pd.DataFrame(b1.reshape(-1, 1), columns=['valor']).to_excel(writer, sheet_name='0_Sesgo_b1')
    pd.DataFrame(W2).to_excel(writer, sheet_name='0_Peso_W2')
    pd.DataFrame(b2.reshape(-1, 1), columns=['valor']).to_excel(writer, sheet_name='0_Sesgo_b2')
    print("✓ Hoja 0: Entradas y Pesos guardados.")

    # --- Fase 1: Forward Pass ---
    pd.DataFrame(Z1.reshape(-1, 1), columns=['valor']).to_excel(writer, sheet_name='1_Forward_Z1')
    pd.DataFrame(A1.reshape(-1, 1), columns=['valor']).to_excel(writer, sheet_name='1_Forward_A1')
    pd.DataFrame(Z2.reshape(-1, 1), columns=['valor']).to_excel(writer, sheet_name='1_Forward_Z2')
    pd.DataFrame(A2.reshape(-1, 1), columns=['probabilidad']).to_excel(writer, sheet_name='1_Forward_A2_Prediccion')
    print("✓ Hoja 1: Resultados del Forward Pass guardados.")

    # --- Fase 2: Error ---
    info_error = {
        'Etiqueta Real': [y_sample],
        'Prediccion': [prediccion],
        'Loss (Cross-Entropy)': [loss]
    }
    pd.DataFrame(info_error).to_excel(writer, sheet_name='2_Error_Loss', index=False)
    pd.DataFrame(y_one_hot.reshape(-1, 1), columns=['valor']).to_excel(writer, sheet_name='2_Error_Y_OneHot')
    print("✓ Hoja 2: Cálculo del Error guardado.")

    # --- Fase 3: Backward Pass (Gradientes) ---
    pd.DataFrame(dZ2.reshape(-1, 1), columns=['gradiente']).to_excel(writer, sheet_name='3_Backward_dZ2')
    pd.DataFrame(dW2).to_excel(writer, sheet_name='3_Backward_dW2')
    pd.DataFrame(db2.reshape(-1, 1), columns=['gradiente']).to_excel(writer, sheet_name='3_Backward_db2')
    pd.DataFrame(dZ1.reshape(-1, 1), columns=['gradiente']).to_excel(writer, sheet_name='3_Backward_dZ1')
    pd.DataFrame(dW1).to_excel(writer, sheet_name='3_Backward_dW1')
    pd.DataFrame(db1.reshape(-1, 1), columns=['gradiente']).to_excel(writer, sheet_name='3_Backward_db1')
    print("✓ Hoja 3: Gradientes del Backward Pass guardados.")

    # --- Fase 4: Actualización de Pesos ---
    pd.DataFrame(W1_new).to_excel(writer, sheet_name='4_Update_W1_new')
    pd.DataFrame(b1_new.reshape(-1, 1), columns=['valor']).to_excel(writer, sheet_name='4_Update_b1_new')
    pd.DataFrame(W2_new).to_excel(writer, sheet_name='4_Update_W2_new')
    pd.DataFrame(b2_new.reshape(-1, 1), columns=['valor']).to_excel(writer, sheet_name='4_Update_b2_new')
    print("✓ Hoja 4: Pesos actualizados guardados.")

print("\n¡Archivo Excel 'calculo_manual.xlsx' creado con éxito!")

Creando archivo Excel en: /home/gerardo/ia/neuronas/document/calculo_manual.xlsx
✓ Hoja 0: Entradas y Pesos guardados.
✓ Hoja 1: Resultados del Forward Pass guardados.
✓ Hoja 2: Cálculo del Error guardado.
✓ Hoja 3: Gradientes del Backward Pass guardados.
✓ Hoja 4: Pesos actualizados guardados.

¡Archivo Excel 'calculo_manual.xlsx' creado con éxito!
